In [ ]:
# ============== COMPLETE DEPENDENCY FIX ==============
print("🔧 Uninstalling ALL old packages...")
# pip uninstall transformers tokenizers accelerate huggingface_hub datasets evaluate pyarrow pandas -y

print("🧹 Clearing ALL caches...")
!pip cache purge
!rm -rf ~/.cache/huggingface
!rm -rf ~/.cache/pip

print("📦 Installing CORRECT compatible versions...")
!pip install transformers==4.37.2 \
    tokenizers==0.15.2 \
    accelerate==0.27.2 \
    peft==0.9.0 \
    datasets==2.14.0 \
    evaluate==0.4.1 \
    pyarrow==14.0.1 \
    pandas==2.1.4 \
    --force-reinstall --no-cache-dir

print("\n✅ Installation complete!")
print("⚠️  NOW RESTART THE RUNTIME (Runtime → Restart Runtime)")
# =========================================================


In [ ]:
!pip uninstall transformers tokenizers accelerate huggingface_hub datasets evaluate pyarrow pandas -y


In [ ]:
import os
os._exit(00)

In [ ]:
# ====================== 
# تثبيت المكتبات المطلوبة
# ======================
!pip install bert_score sacrebleu evaluate scikit-learn joblib rouge_score

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import evaluate
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    GenerationConfig,
    set_seed,
    get_linear_schedule_with_warmup,
    TrainerCallback
)

# ======================
# الإعدادات - 🔧 مُحسّنة
# ======================
MODEL_NAME = "Osame1/AraT5-msa-small-finetuned-xlsum-ar"
DATA_FILE  = "/kaggle/input/datasets/kawkabkawkeb/dataset3/merged_datasetfinalv2 (2).csv"
OUTPUT_DIR = "/kaggle/working/t5-arabic-audience-v5-final"

MAX_INPUT_LENGTH  = 384
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 10

LR = 1e-4
LR_CLASSIFIER = 2e-4
LR_BASE_MULTIPLIER = 0.5

SEED = 42

# 🔧 قيم بيتا المُحسّنة
BETA_SALIENCE = 0.5
BETA_AUDIENCE = 2.0
BETA_DISTANCE = 0.5

WARMUP_RATIO = 0.1
AUDIENCE_TYPES = ["kids", "adult", "expert"]
AUDIENCE_LENGTH_TARGETS = {"kids": 0.5, "adult": 0.75, "expert": 1.0}

LAYERS_TO_UNFREEZE = 4

GENERATION_CONFIG = {
    "num_beams": 4,
    "length_penalty": 1.0,
    "no_repeat_ngram_size": 2,
    "early_stopping": True,
    "do_sample": False,
    "min_length": 32,
    "repetition_penalty": 1.2,
}

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ الجهاز المستخدم: {device}")

# ======================
# تحميل البيانات
# ======================
def load_csv_with_audience(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"خطأ: الملف '{path}' غير موجود.")

    df = pd.read_csv(path)
    df_shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    # df_shuffled=df_shuffled.head(500)
    required_cols = ["text", "summary", "type"]
    missing = [c for c in required_cols if c not in df_shuffled.columns]
    if missing:
        raise ValueError(f"الأعمدة المطلوبة مفقودة: {missing}")

    df_shuffled = df_shuffled[df_shuffled["type"].isin(AUDIENCE_TYPES)].copy()

    data = {"text": [], "summary": [], "audience": []}
    for _, r in tqdm(df_shuffled.iterrows(), total=len(df_shuffled), desc="جاري تحميل البيانات"):
        t, s, a = str(r["text"]).strip(), str(r["summary"]).strip(), str(r["type"]).strip()
        if t and s and a in AUDIENCE_TYPES:
            data["text"].append(t)
            data["summary"].append(s)
            data["audience"].append(a)

    print(f"\n📊 تم تحميل {len(data['text'])} عينة")
    for aud in AUDIENCE_TYPES:
        count = data["audience"].count(aud)
        print(f"  - {aud}: {count} ({count/len(data['text'])*100:.1f}%)")

    return Dataset.from_dict(data), df_shuffled

dataset, original_df = load_csv_with_audience(DATA_FILE)

# ======================
# المعالج النصي ومشفر الجمهور
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

audience_encoder = LabelEncoder()
audience_encoder.fit(AUDIENCE_TYPES)

# 🔧 قوالب أكثر تمييزاً
AUDIENCE_TEMPLATES = {
    "kids": """ملخص للأطفال (عمر 6-12 سنة):
• استخدم كلمات بسيطة جداً
• جمل قصيرة وواضحة (أقل من 15 كلمة)
• نبرة ودية ومبسطة
• ركّز على الفكرة الرئيسية فقط
النص: "{text}"
الملخص للأطفال:""",

    "adult": """لخّص هذا النص لجمهور عام:
• لغة عربية فصحى مبسطة وواضحة
• 3-5 جمل مركزة ومفهومة
• تجنب المصطلحات المعقدة جداً
النص: "{text}"
الملخص للبالغين:""",

    "expert": """ملخص أكاديمي متخصص:
• استخدم مصطلحات تقنية دقيقة
• حافظ على المنهجية والنتائج والإحصائيات
• 4-7 جمل تقنية مكثفة
• لا تُهمل التفاصيل المهمة
النص: "{text}"
الملخص للخبراء:"""
}

def prompt_with_audience(texts, audiences):
    return [AUDIENCE_TEMPLATES[aud].format(text=t) for t, aud in zip(texts, audiences)]

# ======================
# معالجة البيانات - ✅ مُصحّحة
# ======================
def preprocess(examples):
    raw_texts = examples["text"]
    prompted_inputs = prompt_with_audience(examples["text"], examples["audience"])
    targets = examples["summary"]
    audiences = examples["audience"]

    enc_prompted = tokenizer(prompted_inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length")
    enc_raw = tokenizer(raw_texts, max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length")
    
    dec_input_ids = []
    for t, aud in zip(targets, audiences):
        max_len = int(MAX_TARGET_LENGTH * AUDIENCE_LENGTH_TARGETS[aud])
        tokenized = tokenizer(t, max_length=max_len, truncation=True, padding="max_length")
        dec_input_ids.append(tokenized["input_ids"])

    labels, salience, audience_labels = [], [], []
    
    for i, (inp_prompted, lab, aud) in enumerate(zip(enc_prompted["input_ids"], dec_input_ids, audiences)):
        lab_masked = [(t if t != tokenizer.pad_token_id else -100) for t in lab]
        labels.append(lab_masked)
        
        summary_set = set([t for t in lab if t != tokenizer.pad_token_id])
        salience.append([1 if tok in summary_set else 0 for tok in inp_prompted])
        
        audience_labels.append(int(audience_encoder.transform([aud])[0]))

    result = {
        "input_ids": enc_prompted["input_ids"],
        "attention_mask": enc_prompted["attention_mask"],
        "labels": labels,
        "salience_labels": salience,
        "audience_labels": audience_labels,
        "raw_input_ids": enc_raw["input_ids"],
        "raw_attention_mask": enc_raw["attention_mask"],  # ✅ تم التصحيح
    }
    return result

dataset_tok = dataset.map(preprocess, batched=True, remove_columns=["text", "summary", "audience"])
dataset_tok = dataset_tok.train_test_split(test_size=0.1, seed=SEED)

train_ds = dataset_tok["train"]
eval_ds = dataset_tok["test"]

dataset_text_only = dataset.train_test_split(test_size=0.1, seed=SEED)
eval_ds_text = dataset_text_only["test"]

# ======================
# النموذج المُحسّن - ✅ مُصحّح
# ======================
class AudienceAwareT5(nn.Module):
    def __init__(self, name, adapter_dim=256, num_audiences=3):
        super().__init__()
        self.base = AutoModelForSeq2SeqLM.from_pretrained(name)
        d = self.base.config.d_model
        self._keys_to_ignore_on_save = []

        self.audience_embedding = nn.Embedding(num_audiences, d)
        
        self.adapter = nn.Sequential(
            nn.Linear(d, adapter_dim), 
            nn.ReLU(), 
            nn.Linear(adapter_dim, d)
        )
        
        self.sent_proj = nn.Linear(d, d)
        self.salience_head = nn.Linear(d, 1)
        
        self.audience_attention = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.Tanh(),
            nn.Linear(d // 2, 1)
        )
        
        self.audience_classifier = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.ReLU(),
            nn.LayerNorm(d // 2),
            nn.Dropout(0.1),
            nn.Linear(d // 2, d // 4),
            nn.ReLU(),
            nn.LayerNorm(d // 4),
            nn.Linear(d // 4, num_audiences)
        )
        
        # ✅ تم إزالة decoder_audience_proj (كان يسبب خطأ الأبعاد)
        
        self.config = self.base.config
        self.register_buffer("class_weights", torch.ones(num_audiences))
        
        # ✅ تم إزالة decoder_audience_proj من القائمة
        self._custom_modules = ["audience_embedding", "adapter", "sent_proj", 
                                "salience_head", "audience_attention", 
                                "audience_classifier"]
        
    def prepare_inputs_for_generation(self, *args, **kwargs):
        return self.base.prepare_inputs_for_generation(*args, **kwargs)

    def _reorder_cache(self, *args, **kwargs):
        return self.base._reorder_cache(*args, **kwargs)

    @property
    def generation_config(self):
        return self.base.generation_config

    @generation_config.setter
    def generation_config(self, value):
        self.base.generation_config = value

    def sentence_context(self, hidden, attention_mask):
        B, T, D = hidden.shape
        chunk = 20
        ctx = torch.zeros_like(hidden)
        for i in range(0, T, chunk):
            h = hidden[:, i:i+chunk, :]
            m = attention_mask[:, i:i+chunk].unsqueeze(-1)
            pooled = (h * m).sum(1) / (m.sum(1) + 1e-8)
            pooled = self.sent_proj(pooled).unsqueeze(1)
            ctx[:, i:i+chunk, :] = pooled
        return ctx
    
    # ✅ دالة forward مُصحّحة
    def forward(self, input_ids=None, attention_mask=None, labels=None,
                salience_labels=None, audience_labels=None,
                raw_input_ids=None, raw_attention_mask=None, **kwargs):
        
        enc_out = self.base.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            return_dict=True
        )
        H = enc_out.last_hidden_state

        # ✅ زيادة وزن تضمين الجمهور من 0.3 إلى 0.7
        if audience_labels is not None:
            aud_emb = self.audience_embedding(audience_labels)
            aud_emb_seq = aud_emb.unsqueeze(1).expand(-1, H.size(1), -1)
            H = H + aud_emb_seq * 0.7

        H = H + self.sentence_context(H, attention_mask)
        H = H + self.adapter(H)

        salience_logits = self.salience_head(H).squeeze(-1)

        audience_logits = None
        if raw_input_ids is not None and raw_attention_mask is not None:
            raw_enc = self.base.encoder(
                input_ids=raw_input_ids, 
                attention_mask=raw_attention_mask, 
                return_dict=True
            )
            raw_H = raw_enc.last_hidden_state
            cls_representation = raw_H[:, 0, :]
            audience_logits = self.audience_classifier(cls_representation)

        # ✅ تم التصحيح: إزالة decoder_inputs_embeds (كانت تسبب تعارض الأبعاد)
        decoder_outputs = self.base(
            attention_mask=attention_mask,
            encoder_outputs=(H,), 
            labels=labels, 
            return_dict=True,
        )

        total_loss = decoder_outputs.loss if decoder_outputs.loss is not None else torch.tensor(0.0, device=H.device)

        # خسارة الأهمية
        if salience_labels is not None and BETA_SALIENCE > 0 and salience_logits is not None:
            sal_loss = F.binary_cross_entropy_with_logits(
                salience_logits, 
                salience_labels.float(),
                reduction='mean'
            )
            total_loss = total_loss + BETA_SALIENCE * sal_loss

        # خسارة تصنيف الجمهور
        if audience_labels is not None and BETA_AUDIENCE > 0 and audience_logits is not None:
            weights = self.class_weights.to(audience_logits.device)
            aud_loss = F.cross_entropy(
                audience_logits, 
                audience_labels,
                weight=weights,
                reduction='mean'
            )
            total_loss = total_loss + BETA_AUDIENCE * aud_loss

        # ✅ خسارة التباعد بين الجماهير
        if audience_labels is not None and BETA_DISTANCE > 0:
            unique_auds = torch.unique(audience_labels)
            if len(unique_auds) > 1:
                aud_embs = self.audience_embedding(unique_auds)
                aud_embs_norm = F.normalize(aud_embs, p=2, dim=-1)
                sim_matrix = torch.matmul(aud_embs_norm, aud_embs_norm.t())
                target_sim = torch.eye(len(unique_auds), device=sim_matrix.device)
                distance_loss = F.mse_loss(sim_matrix, target_sim)
                total_loss = total_loss + BETA_DISTANCE * distance_loss

        decoder_outputs.loss = total_loss
        decoder_outputs.audience_logits = audience_logits
        decoder_outputs.salience_logits = salience_logits
        return decoder_outputs
        
    def generate(self, *args, **kwargs):
        kwargs.pop('salience_labels', None)
        kwargs.pop('audience_labels', None)
        kwargs.pop('raw_input_ids', None)
        kwargs.pop('raw_attention_mask', None)
        kwargs["use_cache"] = kwargs.get("use_cache", True)
        
        if 'decoder_start_token_id' not in kwargs:
            kwargs['decoder_start_token_id'] = tokenizer.pad_token_id
        if 'eos_token_id' not in kwargs:
            kwargs['eos_token_id'] = tokenizer.eos_token_id
            
        return self.base.generate(*args, **kwargs)

    def predict_audience(self, raw_text, tokenizer, device="cuda"):
        self.eval()
        if isinstance(device, str):
            device = torch.device(device)
        self.to(device)
        
        inputs = tokenizer(
            raw_text, 
            return_tensors="pt", 
            max_length=MAX_INPUT_LENGTH, 
            truncation=True,
            padding="max_length"
        ).to(device)
        
        with torch.no_grad():
            enc = self.base.encoder(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                return_dict=True
            )
            H = enc.last_hidden_state
            cls_rep = H[:, 0, :]
            logits = self.audience_classifier(cls_rep)
            probs = F.softmax(logits, dim=-1)
            pred_idx = torch.argmax(probs, dim=-1)
            
        return pred_idx.item(), probs[0].cpu().numpy()

    def save_pretrained(self, save_directory, **kwargs):
        os.makedirs(save_directory, exist_ok=True)
        self.base.save_pretrained(save_directory, safe_serialization=False)
        
        custom_state = {}
        for name in self._custom_modules:
            module = getattr(self, name)
            custom_state[name] = {k: v.cpu() for k, v in module.state_dict().items()}
        custom_state["class_weights"] = self.class_weights.cpu()
        custom_state["config"] = {
            "adapter_dim": self.adapter[0].in_features,
            "num_audiences": self.audience_classifier[-1].out_features,
        }
        
        torch.save(custom_state, os.path.join(save_directory, "custom_modules.pt"))
        print(f"✅ تم حفظ الوحدات المخصصة في {save_directory}/custom_modules.pt")
        self.config.save_pretrained(save_directory)
        
    @classmethod
    def from_pretrained(cls, save_directory, name=None, **kwargs):
        if name is None:
            config_path = os.path.join(save_directory, "config.json")
            if os.path.exists(config_path):
                from transformers import T5Config
                config = T5Config.from_pretrained(save_directory)
                name = config._name_or_path
        
        model = cls(name or MODEL_NAME, **kwargs)
        model.base = AutoModelForSeq2SeqLM.from_pretrained(save_directory)
        
        custom_path = os.path.join(save_directory, "custom_modules.pt")
        if os.path.exists(custom_path):
            custom_state = torch.load(custom_path, map_location="cpu", weights_only=False)
            for name in model._custom_modules:
                if name in custom_state:
                    getattr(model, name).load_state_dict(custom_state[name])
            if "class_weights" in custom_state:
                model.class_weights = custom_state["class_weights"]
            print(f"✅ تم تحميل الوحدات المخصصة من {custom_path}")
        
        return model

    def unfreeze_top_encoder_layers(self, num_layers=4):
        total_layers = self.base.config.num_layers
        for name, param in self.base.encoder.named_parameters():
            if "layer" in name:
                parts = name.split('.')
                if len(parts) >= 3 and parts[2] == 'block':
                    try:
                        layer_idx = int(parts[3])
                        if layer_idx >= (total_layers - num_layers):
                            param.requires_grad = True
                    except (ValueError, IndexError):
                        continue

    def check_audience_separation(self):
        """فحص التباعد بين تمثيلات الجمهور - نسخة مُصحّحة"""
        aud_embs = self.audience_embedding.weight.detach()  # [3, d_model]
        
        print("\n📏 مصفوفة التشابه/التباعد بين الجماهير:")
        print("-" * 60)
        
        distances = []
        for i, aud1 in enumerate(AUDIENCE_TYPES):
            row_sim = []
            row_dist = []
            for j, aud2 in enumerate(AUDIENCE_TYPES):
                if i == j:
                    row_sim.append(1.000)
                    row_dist.append(0.000)
                else:
                    # حساب التشابه التوافقي الصحيح
                    sim = F.cosine_similarity(
                        aud_embs[i].unsqueeze(0),
                        aud_embs[j].unsqueeze(0),
                        dim=-1
                    ).item()
                    sim = max(-1.0, min(1.0, sim))  # ✅ ضمان النطاق الصحيح
                    dist = 1 - sim  # التباعد = 1 - التشابه
                    
                    row_sim.append(sim)
                    row_dist.append(dist)
                    
                    if i < j:
                        distances.append((aud1, aud2, sim, dist))
            
            # عرض التشابه والتباعد معاً
            sim_str = "  ".join([f"{s:6.3f}" for s in row_sim])
            dist_str = "  ".join([f"{d:6.3f}" for d in row_dist])
            print(f"{aud1:8s} sim: {sim_str}")
            print(f"{'':8s} dist: {dist_str}")
        
        print("-" * 60)
        
        # عرض ملخص التباعد
        print("\n📊 ملخص التباعد (الهدف: ≥ 0.30):")
        avg_dist = 0
        for aud1, aud2, sim, dist in distances:
            status = "✅ جيد" if dist >= 0.30 else "⚠️ ضعيف"
            print(f"  {aud1:8s} vs {aud2:8s}: sim={sim:.3f} | dist={dist:.3f} {status}")
            avg_dist += dist
        
        avg_dist /= len(distances)
        print(f"\n🎯 متوسط التباعد: {avg_dist:.3f} {'✅' if avg_dist >= 0.30 else '⚠️'}")
        
        return avg_dist

# تهيئة النموذج
model = AudienceAwareT5(MODEL_NAME, num_audiences=len(AUDIENCE_TYPES))
model.config.decoder_start_token_id = tokenizer.pad_token_id

# حساب أوزان الفئات
audience_labels_all = audience_encoder.transform(original_df["type"])
class_weights_vals = compute_class_weight(
    'balanced', 
    classes=np.unique(audience_labels_all), 
    y=audience_labels_all
)
model.class_weights = torch.tensor(class_weights_vals, dtype=torch.float32)
print(f"\n🎯 أوزان الفئات: {class_weights_vals}")

# ======================
# جامع البيانات
# ======================
class AudienceAwareCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        salience = [f.pop("salience_labels", None) for f in features]
        audience = [f.pop("audience_labels", None) for f in features]
        raw_input_ids = [f.pop("raw_input_ids", None) for f in features]
        raw_attention_mask = [f.pop("raw_attention_mask", None) for f in features]
        
        batch = super().__call__(features)
        
        if salience[0] is not None:
            batch["salience_labels"] = torch.tensor(salience, dtype=torch.float)
        if audience[0] is not None:
            batch["audience_labels"] = torch.tensor(audience, dtype=torch.long)
        if raw_input_ids[0] is not None:
            batch["raw_input_ids"] = torch.tensor(raw_input_ids, dtype=torch.long)
            batch["raw_attention_mask"] = torch.tensor(raw_attention_mask, dtype=torch.long)
        
        return batch

collator = AudienceAwareCollator(tokenizer, model=model.base, label_pad_token_id=-100)

# ======================
# المقاييس
# ======================
bertscore_metric = evaluate.load("bertscore")
chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    vocab_size = tokenizer.vocab_size
    preds = np.clip(preds.astype(np.int64), 0, vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels.astype(np.int64), 0, vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bertscore = bertscore_metric.compute(predictions=decoded_preds, references=decoded_labels, lang="ar")
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
    bleu = bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bertscore_f1": np.mean(bertscore["f1"]),
        "chrf": chrf["score"],
        "bleu": bleu["bleu"],
        "meteor": meteor["meteor"],
        "rouge1": rouge["rouge1"],
        "rouge2": rouge["rouge2"],
        "rougeL": rouge["rougeL"],
    }

def compute_classifier_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    
    if isinstance(logits, torch.Tensor):
        logits = logits.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    
    preds = np.argmax(logits, axis=-1)
    
    return {
        "audience_accuracy": accuracy_score(labels, preds),
        "audience_f1": f1_score(labels, preds, average="weighted"),
    }

# ======================
# الاستدعاءات
# ======================
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=2, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.wait = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        eval_loss = metrics.get("eval_loss")
        if eval_loss is None:
            return
        if eval_loss < self.best_loss - self.min_delta:
            self.best_loss = eval_loss
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n⚠️ التوقف المبكر بعد {self.patience} عصور بدون تحسن")
                control.should_training_stop = True

# ======================
# إعدادات التدريب
# ======================
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    lr_scheduler_type="linear",
    warmup_ratio=WARMUP_RATIO,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    predict_with_generate=True,
    generation_config=GenerationConfig(**GENERATION_CONFIG),
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    logging_dir=f"{OUTPUT_DIR}/logs",
    save_safetensors=False,
    remove_unused_columns=False,
)

# ======================
# المدرب المخصص
# ======================
class CustomSeq2SeqTrainer(Seq2SeqTrainer):
    def save_model(self, output_dir=None, _internal_call=False):
        if output_dir is None:
            output_dir = self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        if hasattr(self.model, "save_pretrained"):
            self.model.save_pretrained(output_dir)
        else:
            super().save_model(output_dir, _internal_call)
        
        self.tokenizer.save_pretrained(output_dir)
        print(f"✅ تم حفظ النموذج في {output_dir}")
    
    def create_optimizer(self):
        if self.optimizer is None:
            base_params = []
            custom_params = []
            
            for name, param in self.model.named_parameters():
                if not param.requires_grad:
                    continue
                if any(custom in name for custom in self.model._custom_modules):
                    custom_params.append(param)
                else:
                    base_params.append(param)
            
            optimizer_grouped_parameters = [
                {
                    "params": base_params,
                    "lr": self.args.learning_rate * LR_BASE_MULTIPLIER,
                },
                {
                    "params": custom_params,
                    "lr": self.args.learning_rate,
                },
            ]
            
            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(0.9, 0.999),
                eps=1e-8,
                weight_decay=self.args.weight_decay if hasattr(self.args, 'weight_decay') else 0.01,
            )
        return self.optimizer

# ======================
# التدريب على مرحلتين - ✅ مُصحّح
# ======================
def train_stage1_classifier(model, train_ds, eval_ds, epochs=3):
    print("\n" + "="*60)
    print("🎯 المرحلة 1: تدريب مصنّف الجمهور فقط")
    print("="*60)
    
    for name, param in model.base.named_parameters():
        param.requires_grad = False
    
    model.unfreeze_top_encoder_layers(num_layers=LAYERS_TO_UNFREEZE)
    
    for param in model.audience_classifier.parameters():
        param.requires_grad = True
    for param in model.audience_embedding.parameters():
        param.requires_grad = True
    # ✅ تم إزالة decoder_audience_proj
    
    stage1_args = Seq2SeqTrainingArguments(
        output_dir=f"{OUTPUT_DIR}/stage1",
        learning_rate=LR_CLASSIFIER,
        lr_scheduler_type="linear",
        warmup_ratio=WARMUP_RATIO,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=epochs,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=10,
        predict_with_generate=False,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        save_safetensors=False,
        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
    
    stage1_trainer = CustomSeq2SeqTrainer(
        model=model,
        args=stage1_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(patience=2)],
    )
    
    train_result = stage1_trainer.train()
    eval_results = stage1_trainer.evaluate()
    
    print(f"\n📊 نتائج المرحلة 1 النهائية:")
    for k, v in eval_results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
    
    model.check_audience_separation()
    
    print("✅ اكتملت المرحلة 1 - تم تدريب المصنّف")
    return model, stage1_trainer

def train_stage2_joint(model, trainer, train_ds, eval_ds, epochs=3):
    print("\n" + "="*60)
    print("🔄 المرحلة 2: التدريب المشترك (التلخيص + الجمهور)")
    print("="*60)
    
    for name, param in model.base.named_parameters():
        param.requires_grad = True
    
    trainer.args.num_train_epochs = epochs
    trainer.args.learning_rate = LR
    trainer.args.predict_with_generate = True
    trainer.args.compute_metrics = compute_metrics
    trainer.args.load_best_model_at_end = True
    trainer.args.metric_for_best_model = "eval_loss"
    trainer.args.greater_is_better = False
    
    trainer.optimizer = None
    trainer.create_optimizer()
    
    return trainer.train()

# ======================
# بدء التدريب
# ======================
print("\n🏁 بدء التدريب ثنائي المراحل (النسخة المُحسّنة)...")
print(f"📋 الإعدادات: LR={LR}, Classifier LR={LR_CLASSIFIER}, Base LR Multiplier={LR_BASE_MULTIPLIER}")
print(f"📋 قيم بيتا: SALIENCE={BETA_SALIENCE}, AUDIENCE={BETA_AUDIENCE}, DISTANCE={BETA_DISTANCE}")

# المرحلة 1
model, stage1_trainer = train_stage1_classifier(model, train_ds, eval_ds, epochs=3)

# حفظ نقطة التفتيش
stage1_output = f"{OUTPUT_DIR}/stage1-final"
model.save_pretrained(stage1_output)
tokenizer.save_pretrained(stage1_output)
print(f"💾 تم حفظ نقطة تفتيش المرحلة 1 في {stage1_output}")

# =====================================================
# عرض المرحلة 1
# =====================================================
print("\n" + "✨"*30)
print("🔍 عرض المرحلة 1: التنبؤ بنوع الجمهور")
print("✨"*30)

def demo_stage1_predict_audience(model, tokenizer, sample_texts=None):
    model.eval()
    model.to(device)
    
    if sample_texts is None:
        sample_texts = [
            "الشمس نجم يقع في مركز المجموعة الشمسية، وهي مصدر الضوء والحرارة للأرض.",
            "يحتوي الهاتف الذكي على معالج سريع، وشاشة عالية الدقة، وكاميرا متطورة.",
            "تُعد خوارزمية التعلم العميق نوعًا من الشبكات العصبية الاصطناعية."
        ]
    
    print("\n📋 النصوص للتنبؤ بالجمهور:")
    print("-" * 70)
    for i, text in enumerate(sample_texts, 1):
        print(f"\n[{i}] {text[:100]}...")
        pred_idx, probs = model.predict_audience(text, tokenizer, device)
        predicted_audience = AUDIENCE_TYPES[pred_idx]
        
        print(f"🎯 الجمهور المتوقع: {predicted_audience.upper()}")
        print(f"📊 توزيع الثقة:")
        for aud_idx, aud_name in enumerate(AUDIENCE_TYPES):
            bar = "█" * int(probs[aud_idx] * 30)
            print(f"   {aud_name:8s}: {probs[aud_idx]:5.2%} {bar}")
        print("-" * 70)
    
    return sample_texts

demo_stage1_predict_audience(model, tokenizer)

# المرحلة 2
trainer = CustomSeq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(patience=3)],
)

train_stage2_joint(model, trainer, train_ds, eval_ds, epochs=3)

# =====================================================
# عرض المرحلة 2
# =====================================================
print("\n" + "✨"*30)
print("🔄 عرض المرحلة 2: التنبؤ + التلخيص")
print("✨"*30)

def demo_stage2_predict_and_summarize(model, tokenizer, sample_texts=None):
    model.eval()
    model.to(device)
    
    if sample_texts is None:
        sample_texts = [
            "الذكاء الاصطناعي هو فرع من علوم الحاسوب يهدف إلى تطوير أنظمة قادرة على أداء مهام تتطلب ذكاءً بشريًا.",
            "تغير المناخ ظاهرة عالمية ناتجة عن زيادة انبعاثات غازات الدفيئة.",
            "التمثيل الضوئي هو العملية التي تستخدمها النباتات لتحويل الطاقة الضوئية إلى كيميائية."
        ]
    
    print("\n📋 النصوص للتنبؤ والتلخيص:")
    print("=" * 80)
    
    for i, text in enumerate(sample_texts, 1):
        print(f"\n🔹 النص [{i}]:\n{text[:150]}...\n")
        
        pred_idx, probs = model.predict_audience(text, tokenizer, device)
        detected_audience = AUDIENCE_TYPES[pred_idx]
        confidence = probs[pred_idx]
        
        print(f"🎯 الجمهور المكتشف: {detected_audience.upper()} (الثقة: {confidence:.2%})")
        
        prompt = AUDIENCE_TEMPLATES[detected_audience].format(text=text)
        inputs = tokenizer(
            prompt, 
            return_tensors="pt", 
            max_length=MAX_INPUT_LENGTH, 
            truncation=True,
            padding="max_length"
        ).to(device)
        
        gen_max_length = int(MAX_TARGET_LENGTH * AUDIENCE_LENGTH_TARGETS[detected_audience])
        
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids, 
                attention_mask=inputs.attention_mask,
                max_length=gen_max_length,
                **GENERATION_CONFIG,
                decoder_start_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        print(f"\n📝 الملخص ({detected_audience}):")
        print(f"   «{summary}»")
        print("=" * 80)
    
    return sample_texts

demo_stage2_predict_and_summarize(model, tokenizer)

# ======================
# التقييم
# ======================
def evaluate_audience_accuracy(model, tokenizer, audience_encoder, eval_dataset, device="cuda"):
    model.eval()
    if isinstance(device, str):
        device = torch.device(device)
    model.to(device)
    
    predictions, true_labels = [], []
    
    print(f"\n🔍 تقييم دقة التنبؤ بالجمهور على {len(eval_dataset)} عينة...")
    
    for sample in tqdm(eval_dataset, desc="جاري التقييم"):
        text = str(sample["text"]).strip()
        true_audience = str(sample["audience"]).strip()
        
        if not text or true_audience not in AUDIENCE_TYPES:
            continue
            
        pred_idx, probs = model.predict_audience(text, tokenizer, device)
        
        predictions.append(pred_idx)
        true_labels.append(audience_encoder.transform([true_audience])[0])

    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    
    if len(predictions) == 0:
        print("⚠️ لا توجد عينات صالحة للتقييم")
        return 0.0
    
    overall_accuracy = (predictions == true_labels).mean()
    
    print("\n📊 دقة التنبؤ لكل فئة:")
    print("-" * 50)
    for idx, aud_type in enumerate(AUDIENCE_TYPES):
        mask = true_labels == idx
        if mask.sum() > 0:
            acc = (predictions[mask] == idx).mean()
            print(f"  {aud_type:10s}: {acc:6.2%} ({mask.sum():3d} عينة)")
    
    print("-" * 50)
    print(f"✅ الدقة الإجمالية: {overall_accuracy:.2%}")
    
    print("\n📋 تقرير التصنيف:")
    print(classification_report(true_labels, predictions, target_names=AUDIENCE_TYPES, digits=4))
    
    cm = confusion_matrix(true_labels, predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=AUDIENCE_TYPES, yticklabels=AUDIENCE_TYPES)
    plt.title('مصفوفة الارتباك - التنبؤ بالجمهور')
    plt.ylabel('الفعلي')
    plt.xlabel('المتوقع')
    plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png')
    plt.close()
    print(f"\n💾 تم حفظ مصفوفة الارتباك في {OUTPUT_DIR}/confusion_matrix.png")
    
    return overall_accuracy

audience_acc = evaluate_audience_accuracy(
    model, 
    tokenizer, 
    audience_encoder, 
    eval_dataset=eval_ds_text,
    device=device
)

# ======================
# المقاييس النهائية
# ======================
print("\n" + "="*60)
print("📈 مقاييس التلخيص النهائية")
print("="*60)
metrics = trainer.evaluate()

if "eval_loss" in metrics and metrics["eval_loss"] is not None:
    metrics["perplexity"] = math.exp(metrics["eval_loss"])

for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")
print("="*60)

# ======================
# حفظ النموذج النهائي
# ======================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
joblib.dump(audience_encoder, os.path.join(OUTPUT_DIR, "audience_encoder.pkl"))

print(f"\n✅ تم حفظ النموذج والبيانات في: {OUTPUT_DIR}")

# فحص نهائي لتباعد الجمهور
print("\n" + "="*60)
print("📏 الفحص النهائي لتباعد تمثيلات الجمهور")
print("="*60)
final_separation = model.check_audience_separation()

# ======================
# دوال الاستدلال
# ======================
def summarize_for_audience(text, audience_type="adult", model=None, tokenizer=None, device="cuda"):
    if audience_type not in AUDIENCE_TYPES:
        raise ValueError(f"الجمهور يجب أن يكون أحد: {AUDIENCE_TYPES}")
    
    if isinstance(device, str):
        device = torch.device(device)
    model.eval()
    model.to(device)
    
    prompt = AUDIENCE_TEMPLATES[audience_type].format(text=text)
    
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        max_length=MAX_INPUT_LENGTH, 
        truncation=True,
        padding="max_length"
    ).to(device)
    
    gen_max_length = int(MAX_TARGET_LENGTH * AUDIENCE_LENGTH_TARGETS[audience_type])
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids, 
            attention_mask=inputs.attention_mask,
            max_length=gen_max_length,
            **GENERATION_CONFIG,
            decoder_start_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def summarize_with_audience_detection(text, model=None, tokenizer=None, device="cuda"):
    if isinstance(device, str):
        device = torch.device(device)
    model.eval()
    model.to(device)
    
    pred_idx, probs = model.predict_audience(text, tokenizer, device)
    detected_audience = AUDIENCE_TYPES[pred_idx]
    confidence = probs[pred_idx]
    
    summary = summarize_for_audience(
        text, 
        audience_type=detected_audience, 
        model=model, 
        tokenizer=tokenizer, 
        device=device
    )
    
    return {
        "summary": summary,
        "detected_audience": detected_audience,
        "confidence": float(confidence),
        "audience_distribution": {
            aud: float(probs[i]) for i, aud in enumerate(AUDIENCE_TYPES)
        }
    }

# ======================
# الخاتمة
# ======================
print("\n" + "✨"*30)
print("🎉 اكتمل التدريب بنجاح!")
print("✨"*30)
print(f"\n📊 الدقة النهائية للجمهور: {audience_acc:.2%}")
print(f"📏 متوسط تباعد الجمهور: {final_separation:.3f}")
print(f"🎯 الهدف: دقة ≥90% | تباعد ≥0.3")
print(f"{'✅ تم تحقيق الهدف' if audience_acc >= 0.90 and final_separation >= 0.3 else '🔄 يحتاج لمزيد من الضبط'}")

print("\n📚 أمثلة الاستخدام:")
print('   # التنبؤ بالجمهور فقط:')
print('   pred_idx, probs = model.predict_audience(text, tokenizer)')
print()
print('   # التلخيص مع التنبؤ:')
print('   result = summarize_with_audience_detection(text, model, tokenizer)')
print('   print(result["summary"])')
print()
print('   # التلخيص لجمهور محدد:')
print('   summary = summarize_for_audience(text, "kids", model, tokenizer)')

In [ ]:
# ======================
# 🧪 اختبار النموذج وحفظ النتائج
# ======================
import pandas as pd
import torch
from tqdm import tqdm

# التأكد من أن النموذج في وضع التقييم
model.eval()
model.to(device)

# إعداد قائمة لتخزين النتائج
results = []

# الحصول على مجموعة الاختبار (نفس الـ 10% المستخدمة في التدريب)
test_samples = eval_ds_text  # هذه تحتوي على النص الأصلي + النوع

print(f"\n🔍 جاري الاختبار على {len(test_samples)} عينة...")

for idx in tqdm(range(len(test_samples)), desc="جاري التنبؤ"):
    sample = test_samples[idx]
    
    original_text = str(sample["text"]).strip()
    true_summary = str(sample["summary"]).strip()
    audience_type = str(sample["audience"]).strip()
    
    if not original_text or audience_type not in AUDIENCE_TYPES:
        continue
    
    # 🔹 الخطوة 1: التنبؤ بنوع الجمهور (اختياري - للمقارنة)
    pred_idx, probs = model.predict_audience(original_text, tokenizer, device)
    predicted_audience = AUDIENCE_TYPES[pred_idx]
    confidence = probs[pred_idx]
    
    # 🔹 الخطوة 2: توليد الملخص بناءً على نوع الجمهور (المحدد في البيانات أو المتوقع)
    # نستخدم النوع الحقيقي من البيانات لضمان تقييم عادل
    target_audience = audience_type  # أو استخدم predicted_audience إذا أردت اختبار التنبؤ
    
    prompt = AUDIENCE_TEMPLATES[target_audience].format(text=original_text)
    
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        max_length=MAX_INPUT_LENGTH, 
        truncation=True,
        padding="max_length"
    ).to(device)
    
    gen_max_length = int(MAX_TARGET_LENGTH * AUDIENCE_LENGTH_TARGETS[target_audience])
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids, 
            attention_mask=inputs.attention_mask,
            max_length=gen_max_length,
            **GENERATION_CONFIG,
            decoder_start_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    predicted_summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # 🔹 حفظ النتيجة
    results.append({
        "text": original_text,
        "true_summary": true_summary,
        "predicted_summary": predicted_summary,
        "audience_type": audience_type,  # النوع الحقيقي من البيانات
        "predicted_audience": predicted_audience,  # النوع الذي تنبأ به النموذج
        "confidence": round(float(confidence), 4),
        "match": audience_type == predicted_audience  # هل تطابق التنبؤ مع الواقع؟
    })

# ======================
# 💾 حفظ النتائج في ملف CSV
# ======================
results_df = pd.DataFrame(results)

# حفظ الملف الكامل
output_path = os.path.join(OUTPUT_DIR, "test_results_with_audience.csv")
results_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n✅ تم حفظ النتائج في: {output_path}")

# حفظ نسخة مختصرة (كما طلبت: text, predicted-summary, type)
simple_results = results_df[["text", "predicted_summary", "audience_type"]].copy()
simple_results.columns = ["text", "predicted-summary", "type"]  # إعادة تسمية الأعمدة كما طلبت

simple_output_path = os.path.join(OUTPUT_DIR, "test_results_simple.csv")
simple_results.to_csv(simple_output_path, index=False, encoding="utf-8-sig")
print(f"✅ تم حفظ النتائج المبسطة في: {simple_output_path}")

# ======================
# 📊 عرض إحصائيات سريعة
# ======================
print("\n" + "="*60)
print("📈 إحصائيات النتائج")
print("="*60)

print(f"\n📋 عدد العينات المُعالَجة: {len(results_df)}")

# دقة التنبؤ بنوع الجمهور
if "match" in results_df.columns:
    audience_accuracy = results_df["match"].mean()
    print(f"🎯 دقة التنبؤ بنوع الجمهور: {audience_accuracy:.2%}")

# توزيع النتائج حسب نوع الجمهور
print(f"\n📊 التوزيع حسب نوع الجمهور:")
for aud in AUDIENCE_TYPES:
    count = len(results_df[results_df["audience_type"] == aud])
    print(f"  - {aud}: {count} عينة ({count/len(results_df)*100:.1f}%)")

# معاينة أول 3 نتائج
print(f"\n🔍 معاينة أول 3 نتائج:")
print("-" * 80)
for i, row in results_df.head(3).iterrows():
    print(f"\n[{i+1}] الجمهور: {row['audience_type']} | الثقة: {row['confidence']:.2%}")
    print(f"📝 النص: {row['text'][:150]}...")
    print(f"✨ الملخص المتوقع: {row['predicted_summary']}")
    print("-" * 80)

In [ ]:
# ======================
# INSTALL (if needed)
# ======================
# !pip install transformers sentencepiece torch pandas

import os
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ======================
# CONFIG
# ======================
MODEL_NAME = "Osame1/AraT5-msa-small-finetuned-xlsum-ar"
CSV_PATH   = "/kaggle/input/datasets/kawkabkawkeb/testing/test_results_with_audience.csv"
OUTPUT_DIR = "/kaggle/working/t5-arabic-without_finetuning"

MAX_INPUT_LENGTH = 384
MAX_OUTPUT_LENGTH = 96
BATCH_SIZE = 8

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# DEVICE
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# LOAD MODEL
# ======================
print(f"📥 Loading model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

print(f"✅ Model loaded on {device}")

# ======================
# LOAD CSV
# ======================
df = pd.read_csv(CSV_PATH)

if "text" not in df.columns:
    raise ValueError("CSV must contain 'text' column")

texts = df["text"].astype(str).fillna("").tolist()

# ======================
# GENERATE FUNCTION (BATCH)
# ======================
def generate_batch(texts_batch):
    inputs = tokenizer(
        [f"النص: {t}\nالملخص:" for t in texts_batch],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_OUTPUT_LENGTH,
            num_beams=5,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
            use_cache=False
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# ======================
# RUN INFERENCE
# ======================
generated_summaries = []

print("\n🚀 Generating summaries...\n")

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i+BATCH_SIZE]

    summaries = generate_batch(batch)
    generated_summaries.extend(summaries)

# ======================
# SAVE RESULTS
# ======================
df["generated_summary"] = generated_summaries

OUTPUT_FILE = os.path.join(OUTPUT_DIR, "generated_summaries_zero.csv")
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"\n✅ Done! Results saved to: {OUTPUT_FILE}")

In [ ]:
# ====================== 
# تثبيت المكتبات المطلوبة
# ======================
!pip install bert_score sacrebleu evaluate scikit-learn joblib rouge_score

import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import evaluate
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import joblib
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    GenerationConfig,
    set_seed,
    get_linear_schedule_with_warmup,
    TrainerCallback
)

# ======================
# الإعدادات - 🔧 مُحسّنة
# ======================
MODEL_NAME = "Osame1/AraT5-msa-small-finetuned-xlsum-ar"
DATA_FILE  = "/kaggle/input/datasets/kawkabkawkeb/dataset3/merged_datasetfinalv2 (2).csv"
OUTPUT_DIR = "/kaggle/working/t5-arabic-audience-v5-final"

MAX_INPUT_LENGTH  = 384
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 10

LR = 1e-4
LR_CLASSIFIER = 2e-4
LR_BASE_MULTIPLIER = 0.5

SEED = 42

# 🔧 قيم بيتا المُحسّنة
BETA_SALIENCE = 0.5
BETA_AUDIENCE = 2.0
BETA_DISTANCE = 0.5

WARMUP_RATIO = 0.1
AUDIENCE_TYPES = ["kids", "adult", "expert"]
AUDIENCE_LENGTH_TARGETS = {"kids": 0.5, "adult": 0.75, "expert": 1.0}

LAYERS_TO_UNFREEZE = 4

GENERATION_CONFIG = {
    "num_beams": 4,
    "length_penalty": 1.0,
    "no_repeat_ngram_size": 2,
    "early_stopping": True,
    "do_sample": False,
    "min_length": 32,
    "repetition_penalty": 1.2,
}

set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ الجهاز المستخدم: {device}")

# ======================
# تحميل البيانات
# ======================
def load_csv_with_audience(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"خطأ: الملف '{path}' غير موجود.")

    df = pd.read_csv(path)
    df_shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    # df_shuffled=df_shuffled.head(500)
    required_cols = ["text", "summary", "type"]
    missing = [c for c in required_cols if c not in df_shuffled.columns]
    if missing:
        raise ValueError(f"الأعمدة المطلوبة مفقودة: {missing}")

    df_shuffled = df_shuffled[df_shuffled["type"].isin(AUDIENCE_TYPES)].copy()

    data = {"text": [], "summary": [], "audience": []}
    for _, r in tqdm(df_shuffled.iterrows(), total=len(df_shuffled), desc="جاري تحميل البيانات"):
        t, s, a = str(r["text"]).strip(), str(r["summary"]).strip(), str(r["type"]).strip()
        if t and s and a in AUDIENCE_TYPES:
            data["text"].append(t)
            data["summary"].append(s)
            data["audience"].append(a)

    print(f"\n📊 تم تحميل {len(data['text'])} عينة")
    for aud in AUDIENCE_TYPES:
        count = data["audience"].count(aud)
        print(f"  - {aud}: {count} ({count/len(data['text'])*100:.1f}%)")

    return Dataset.from_dict(data), df_shuffled

dataset, original_df = load_csv_with_audience(DATA_FILE)

# ======================
# المعالج النصي ومشفر الجمهور
# ======================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0

audience_encoder = LabelEncoder()
audience_encoder.fit(AUDIENCE_TYPES)

# 🔧 قوالب أكثر تمييزاً
AUDIENCE_TEMPLATES = {
    "kids": """ملخص للأطفال (عمر 6-12 سنة):
• استخدم كلمات بسيطة جداً
• جمل قصيرة وواضحة (أقل من 15 كلمة)
• نبرة ودية ومبسطة
• ركّز على الفكرة الرئيسية فقط
النص: "{text}"
الملخص للأطفال:""",

    "adult": """لخّص هذا النص لجمهور عام:
• لغة عربية فصحى مبسطة وواضحة
• 3-5 جمل مركزة ومفهومة
• تجنب المصطلحات المعقدة جداً
النص: "{text}"
الملخص للبالغين:""",

    "expert": """ملخص أكاديمي متخصص:
• استخدم مصطلحات تقنية دقيقة
• حافظ على المنهجية والنتائج والإحصائيات
• 4-7 جمل تقنية مكثفة
• لا تُهمل التفاصيل المهمة
النص: "{text}"
الملخص للخبراء:"""
}

def prompt_with_audience(texts, audiences):
    return [AUDIENCE_TEMPLATES[aud].format(text=t) for t, aud in zip(texts, audiences)]

# ======================
# معالجة البيانات - ✅ مُصحّحة
# ======================
def preprocess(examples):
    raw_texts = examples["text"]
    prompted_inputs = prompt_with_audience(examples["text"], examples["audience"])
    targets = examples["summary"]
    audiences = examples["audience"]

    enc_prompted = tokenizer(prompted_inputs, max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length")
    enc_raw = tokenizer(raw_texts, max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length")
    
    dec_input_ids = []
    for t, aud in zip(targets, audiences):
        max_len = int(MAX_TARGET_LENGTH * AUDIENCE_LENGTH_TARGETS[aud])
        tokenized = tokenizer(t, max_length=max_len, truncation=True, padding="max_length")
        dec_input_ids.append(tokenized["input_ids"])

    labels, salience, audience_labels = [], [], []
    
    for i, (inp_prompted, lab, aud) in enumerate(zip(enc_prompted["input_ids"], dec_input_ids, audiences)):
        lab_masked = [(t if t != tokenizer.pad_token_id else -100) for t in lab]
        labels.append(lab_masked)
        
        summary_set = set([t for t in lab if t != tokenizer.pad_token_id])
        salience.append([1 if tok in summary_set else 0 for tok in inp_prompted])
        
        audience_labels.append(int(audience_encoder.transform([aud])[0]))

    result = {
        "input_ids": enc_prompted["input_ids"],
        "attention_mask": enc_prompted["attention_mask"],
        "labels": labels,
        "salience_labels": salience,
        "audience_labels": audience_labels,
        "raw_input_ids": enc_raw["input_ids"],
        "raw_attention_mask": enc_raw["attention_mask"],  # ✅ تم التصحيح
    }
    return result

dataset_tok = dataset.map(preprocess, batched=True, remove_columns=["text", "summary", "audience"])
dataset_tok = dataset_tok.train_test_split(test_size=0.1, seed=SEED)

train_ds = dataset_tok["train"]
eval_ds = dataset_tok["test"]

dataset_text_only = dataset.train_test_split(test_size=0.1, seed=SEED)
eval_ds_text = dataset_text_only["test"]

# ======================
# النموذج المُحسّن - ✅ مُصحّح
# ======================
class AudienceAwareT5(nn.Module):
    def __init__(self, name, adapter_dim=256, num_audiences=3):
        super().__init__()
        self.base = AutoModelForSeq2SeqLM.from_pretrained(name)
        d = self.base.config.d_model
        self._keys_to_ignore_on_save = []

        self.audience_embedding = nn.Embedding(num_audiences, d)
        
        self.adapter = nn.Sequential(
            nn.Linear(d, adapter_dim), 
            nn.ReLU(), 
            nn.Linear(adapter_dim, d)
        )
        
        self.sent_proj = nn.Linear(d, d)
        self.salience_head = nn.Linear(d, 1)
        
        self.audience_attention = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.Tanh(),
            nn.Linear(d // 2, 1)
        )
        
        self.audience_classifier = nn.Sequential(
            nn.Linear(d, d // 2),
            nn.ReLU(),
            nn.LayerNorm(d // 2),
            nn.Dropout(0.1),
            nn.Linear(d // 2, d // 4),
            nn.ReLU(),
            nn.LayerNorm(d // 4),
            nn.Linear(d // 4, num_audiences)
        )
        
        # ✅ تم إزالة decoder_audience_proj (كان يسبب خطأ الأبعاد)
        
        self.config = self.base.config
        self.register_buffer("class_weights", torch.ones(num_audiences))
        
        # ✅ تم إزالة decoder_audience_proj من القائمة
        self._custom_modules = ["audience_embedding", "adapter", "sent_proj", 
                                "salience_head", "audience_attention", 
                                "audience_classifier"]
        
    def prepare_inputs_for_generation(self, *args, **kwargs):
        return self.base.prepare_inputs_for_generation(*args, **kwargs)

    def _reorder_cache(self, *args, **kwargs):
        return self.base._reorder_cache(*args, **kwargs)

    @property
    def generation_config(self):
        return self.base.generation_config

    @generation_config.setter
    def generation_config(self, value):
        self.base.generation_config = value

    def sentence_context(self, hidden, attention_mask):
        B, T, D = hidden.shape
        chunk = 20
        ctx = torch.zeros_like(hidden)
        for i in range(0, T, chunk):
            h = hidden[:, i:i+chunk, :]
            m = attention_mask[:, i:i+chunk].unsqueeze(-1)
            pooled = (h * m).sum(1) / (m.sum(1) + 1e-8)
            pooled = self.sent_proj(pooled).unsqueeze(1)
            ctx[:, i:i+chunk, :] = pooled
        return ctx
    
    # ✅ دالة forward مُصحّحة
    def forward(self, input_ids=None, attention_mask=None, labels=None,
                salience_labels=None, audience_labels=None,
                raw_input_ids=None, raw_attention_mask=None, **kwargs):
        
        enc_out = self.base.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            return_dict=True
        )
        H = enc_out.last_hidden_state

        # ✅ زيادة وزن تضمين الجمهور من 0.3 إلى 0.7
        if audience_labels is not None:
            aud_emb = self.audience_embedding(audience_labels)
            aud_emb_seq = aud_emb.unsqueeze(1).expand(-1, H.size(1), -1)
            H = H + aud_emb_seq * 0.7

        H = H + self.sentence_context(H, attention_mask)
        H = H + self.adapter(H)

        salience_logits = self.salience_head(H).squeeze(-1)

        audience_logits = None
        if raw_input_ids is not None and raw_attention_mask is not None:
            raw_enc = self.base.encoder(
                input_ids=raw_input_ids, 
                attention_mask=raw_attention_mask, 
                return_dict=True
            )
            raw_H = raw_enc.last_hidden_state
            cls_representation = raw_H[:, 0, :]
            audience_logits = self.audience_classifier(cls_representation)

        # ✅ تم التصحيح: إزالة decoder_inputs_embeds (كانت تسبب تعارض الأبعاد)
        decoder_outputs = self.base(
            attention_mask=attention_mask,
            encoder_outputs=(H,), 
            labels=labels, 
            return_dict=True,
        )

        total_loss = decoder_outputs.loss if decoder_outputs.loss is not None else torch.tensor(0.0, device=H.device)

        # خسارة الأهمية
        if salience_labels is not None and BETA_SALIENCE > 0 and salience_logits is not None:
            sal_loss = F.binary_cross_entropy_with_logits(
                salience_logits, 
                salience_labels.float(),
                reduction='mean'
            )
            total_loss = total_loss + BETA_SALIENCE * sal_loss

        # خسارة تصنيف الجمهور
        if audience_labels is not None and BETA_AUDIENCE > 0 and audience_logits is not None:
            weights = self.class_weights.to(audience_logits.device)
            aud_loss = F.cross_entropy(
                audience_logits, 
                audience_labels,
                weight=weights,
                reduction='mean'
            )
            total_loss = total_loss + BETA_AUDIENCE * aud_loss

        # ✅ خسارة التباعد بين الجماهير
        if audience_labels is not None and BETA_DISTANCE > 0:
            unique_auds = torch.unique(audience_labels)
            if len(unique_auds) > 1:
                aud_embs = self.audience_embedding(unique_auds)
                aud_embs_norm = F.normalize(aud_embs, p=2, dim=-1)
                sim_matrix = torch.matmul(aud_embs_norm, aud_embs_norm.t())
                target_sim = torch.eye(len(unique_auds), device=sim_matrix.device)
                distance_loss = F.mse_loss(sim_matrix, target_sim)
                total_loss = total_loss + BETA_DISTANCE * distance_loss

        decoder_outputs.loss = total_loss
        decoder_outputs.audience_logits = audience_logits
        decoder_outputs.salience_logits = salience_logits
        return decoder_outputs
        
    def generate(self, *args, **kwargs):
        kwargs.pop('salience_labels', None)
        kwargs.pop('audience_labels', None)
        kwargs.pop('raw_input_ids', None)
        kwargs.pop('raw_attention_mask', None)
        kwargs["use_cache"] = kwargs.get("use_cache", True)
        
        if 'decoder_start_token_id' not in kwargs:
            kwargs['decoder_start_token_id'] = tokenizer.pad_token_id
        if 'eos_token_id' not in kwargs:
            kwargs['eos_token_id'] = tokenizer.eos_token_id
            
        return self.base.generate(*args, **kwargs)

    def predict_audience(self, raw_text, tokenizer, device="cuda"):
        self.eval()
        if isinstance(device, str):
            device = torch.device(device)
        self.to(device)
        
        inputs = tokenizer(
            raw_text, 
            return_tensors="pt", 
            max_length=MAX_INPUT_LENGTH, 
            truncation=True,
            padding="max_length"
        ).to(device)
        
        with torch.no_grad():
            enc = self.base.encoder(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                return_dict=True
            )
            H = enc.last_hidden_state
            cls_rep = H[:, 0, :]
            logits = self.audience_classifier(cls_rep)
            probs = F.softmax(logits, dim=-1)
            pred_idx = torch.argmax(probs, dim=-1)
            
        return pred_idx.item(), probs[0].cpu().numpy()

    def save_pretrained(self, save_directory, **kwargs):
        os.makedirs(save_directory, exist_ok=True)
        self.base.save_pretrained(save_directory, safe_serialization=False)
        
        custom_state = {}
        for name in self._custom_modules:
            module = getattr(self, name)
            custom_state[name] = {k: v.cpu() for k, v in module.state_dict().items()}
        custom_state["class_weights"] = self.class_weights.cpu()
        custom_state["config"] = {
            "adapter_dim": self.adapter[0].in_features,
            "num_audiences": self.audience_classifier[-1].out_features,
        }
        
        torch.save(custom_state, os.path.join(save_directory, "custom_modules.pt"))
        print(f"✅ تم حفظ الوحدات المخصصة في {save_directory}/custom_modules.pt")
        self.config.save_pretrained(save_directory)
        
    @classmethod
    def from_pretrained(cls, save_directory, name=None, **kwargs):
        if name is None:
            config_path = os.path.join(save_directory, "config.json")
            if os.path.exists(config_path):
                from transformers import T5Config
                config = T5Config.from_pretrained(save_directory)
                name = config._name_or_path
        
        model = cls(name or MODEL_NAME, **kwargs)
        model.base = AutoModelForSeq2SeqLM.from_pretrained(save_directory)
        
        custom_path = os.path.join(save_directory, "custom_modules.pt")
        if os.path.exists(custom_path):
            custom_state = torch.load(custom_path, map_location="cpu", weights_only=False)
            for name in model._custom_modules:
                if name in custom_state:
                    getattr(model, name).load_state_dict(custom_state[name])
            if "class_weights" in custom_state:
                model.class_weights = custom_state["class_weights"]
            print(f"✅ تم تحميل الوحدات المخصصة من {custom_path}")
        
        return model

    def unfreeze_top_encoder_layers(self, num_layers=4):
        total_layers = self.base.config.num_layers
        for name, param in self.base.encoder.named_parameters():
            if "layer" in name:
                parts = name.split('.')
                if len(parts) >= 3 and parts[2] == 'block':
                    try:
                        layer_idx = int(parts[3])
                        if layer_idx >= (total_layers - num_layers):
                            param.requires_grad = True
                    except (ValueError, IndexError):
                        continue

    def check_audience_separation(self):
        """فحص التباعد بين تمثيلات الجمهور - نسخة مُصحّحة"""
        aud_embs = self.audience_embedding.weight.detach()  # [3, d_model]
        
        print("\n📏 مصفوفة التشابه/التباعد بين الجماهير:")
        print("-" * 60)
        
        distances = []
        for i, aud1 in enumerate(AUDIENCE_TYPES):
            row_sim = []
            row_dist = []
            for j, aud2 in enumerate(AUDIENCE_TYPES):
                if i == j:
                    row_sim.append(1.000)
                    row_dist.append(0.000)
                else:
                    # حساب التشابه التوافقي الصحيح
                    sim = F.cosine_similarity(
                        aud_embs[i].unsqueeze(0),
                        aud_embs[j].unsqueeze(0),
                        dim=-1
                    ).item()
                    sim = max(-1.0, min(1.0, sim))  # ✅ ضمان النطاق الصحيح
                    dist = 1 - sim  # التباعد = 1 - التشابه
                    
                    row_sim.append(sim)
                    row_dist.append(dist)
                    
                    if i < j:
                        distances.append((aud1, aud2, sim, dist))
            
            # عرض التشابه والتباعد معاً
            sim_str = "  ".join([f"{s:6.3f}" for s in row_sim])
            dist_str = "  ".join([f"{d:6.3f}" for d in row_dist])
            print(f"{aud1:8s} sim: {sim_str}")
            print(f"{'':8s} dist: {dist_str}")
        
        print("-" * 60)
        
        # عرض ملخص التباعد
        print("\n📊 ملخص التباعد (الهدف: ≥ 0.30):")
        avg_dist = 0
        for aud1, aud2, sim, dist in distances:
            status = "✅ جيد" if dist >= 0.30 else "⚠️ ضعيف"
            print(f"  {aud1:8s} vs {aud2:8s}: sim={sim:.3f} | dist={dist:.3f} {status}")
            avg_dist += dist
        
        avg_dist /= len(distances)
        print(f"\n🎯 متوسط التباعد: {avg_dist:.3f} {'✅' if avg_dist >= 0.30 else '⚠️'}")
        
        return avg_dist

# تهيئة النموذج
model = AudienceAwareT5(MODEL_NAME, num_audiences=len(AUDIENCE_TYPES))
model.config.decoder_start_token_id = tokenizer.pad_token_id

# حساب أوزان الفئات
audience_labels_all = audience_encoder.transform(original_df["type"])
class_weights_vals = compute_class_weight(
    'balanced', 
    classes=np.unique(audience_labels_all), 
    y=audience_labels_all
)
model.class_weights = torch.tensor(class_weights_vals, dtype=torch.float32)
print(f"\n🎯 أوزان الفئات: {class_weights_vals}")

# ======================
# جامع البيانات
# ======================
class AudienceAwareCollator(DataCollatorForSeq2Seq):
    def __call__(self, features):
        salience = [f.pop("salience_labels", None) for f in features]
        audience = [f.pop("audience_labels", None) for f in features]
        raw_input_ids = [f.pop("raw_input_ids", None) for f in features]
        raw_attention_mask = [f.pop("raw_attention_mask", None) for f in features]
        
        batch = super().__call__(features)
        
        if salience[0] is not None:
            batch["salience_labels"] = torch.tensor(salience, dtype=torch.float)
        if audience[0] is not None:
            batch["audience_labels"] = torch.tensor(audience, dtype=torch.long)
        if raw_input_ids[0] is not None:
            batch["raw_input_ids"] = torch.tensor(raw_input_ids, dtype=torch.long)
            batch["raw_attention_mask"] = torch.tensor(raw_attention_mask, dtype=torch.long)
        
        return batch

collator = AudienceAwareCollator(tokenizer, model=model.base, label_pad_token_id=-100)

# ======================
# المقاييس
# ======================
bertscore_metric = evaluate.load("bertscore")
chrf_metric = evaluate.load("chrf")
bleu_metric = evaluate.load("bleu")
meteor_metric = evaluate.load("meteor")
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    vocab_size = tokenizer.vocab_size
    preds = np.clip(preds.astype(np.int64), 0, vocab_size - 1)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.clip(labels.astype(np.int64), 0, vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bertscore = bertscore_metric.compute(predictions=decoded_preds, references=decoded_labels, lang="ar")
    chrf = chrf_metric.compute(predictions=decoded_preds, references=decoded_labels)
    bleu = bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    meteor = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bertscore_f1": np.mean(bertscore["f1"]),
        "chrf": chrf["score"],
        "bleu": bleu["bleu"],
        "meteor": meteor["meteor"],
        "rouge1": rouge["rouge1"],
        "rouge2": rouge["rouge2"],
        "rougeL": rouge["rougeL"],
    }

def compute_classifier_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    
    if isinstance(logits, torch.Tensor):
        logits = logits.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()
    
    preds = np.argmax(logits, axis=-1)
    
    return {
        "audience_accuracy": accuracy_score(labels, preds),
        "audience_f1": f1_score(labels, preds, average="weighted"),
    }

# ======================
# الاستدعاءات
# ======================
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience=2, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.wait = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        eval_loss = metrics.get("eval_loss")
        if eval_loss is None:
            return
        if eval_loss < self.best_loss - self.min_delta:
            self.best_loss = eval_loss
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n⚠️ التوقف المبكر بعد {self.patience} عصور بدون تحسن")
                control.should_training_stop = True

# ======================
# إعدادات التدريب
# ======================
args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    lr_scheduler_type="linear",
    warmup_ratio=WARMUP_RATIO,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    predict_with_generate=True,
    generation_config=GenerationConfig(**GENERATION_CONFIG),
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    logging_dir=f"{OUTPUT_DIR}/logs",
    save_safetensors=False,
    remove_unused_columns=False,
)

# ======================
# المدرب المخصص
# ======================
class CustomSeq2SeqTrainer(Seq2SeqTrainer):
    def save_model(self, output_dir=None, _internal_call=False):
        if output_dir is None:
            output_dir = self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        if hasattr(self.model, "save_pretrained"):
            self.model.save_pretrained(output_dir)
        else:
            super().save_model(output_dir, _internal_call)
        
        self.tokenizer.save_pretrained(output_dir)
        print(f"✅ تم حفظ النموذج في {output_dir}")
    
    def create_optimizer(self):
        if self.optimizer is None:
            base_params = []
            custom_params = []
            
            for name, param in self.model.named_parameters():
                if not param.requires_grad:
                    continue
                if any(custom in name for custom in self.model._custom_modules):
                    custom_params.append(param)
                else:
                    base_params.append(param)
            
            optimizer_grouped_parameters = [
                {
                    "params": base_params,
                    "lr": self.args.learning_rate * LR_BASE_MULTIPLIER,
                },
                {
                    "params": custom_params,
                    "lr": self.args.learning_rate,
                },
            ]
            
            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                lr=self.args.learning_rate,
                betas=(0.9, 0.999),
                eps=1e-8,
                weight_decay=self.args.weight_decay if hasattr(self.args, 'weight_decay') else 0.01,
            )
        return self.optimizer

# ======================
# التدريب على مرحلتين - ✅ مُصحّح
# ======================
def train_stage1_classifier(model, train_ds, eval_ds, epochs=3):
    print("\n" + "="*60)
    print("🎯 المرحلة 1: تدريب مصنّف الجمهور فقط")
    print("="*60)
    
    for name, param in model.base.named_parameters():
        param.requires_grad = False
    
    model.unfreeze_top_encoder_layers(num_layers=LAYERS_TO_UNFREEZE)
    
    for param in model.audience_classifier.parameters():
        param.requires_grad = True
    for param in model.audience_embedding.parameters():
        param.requires_grad = True
    # ✅ تم إزالة decoder_audience_proj
    
    stage1_args = Seq2SeqTrainingArguments(
        output_dir=f"{OUTPUT_DIR}/stage1",
        learning_rate=LR_CLASSIFIER,
        lr_scheduler_type="linear",
        warmup_ratio=WARMUP_RATIO,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=epochs,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=10,
        predict_with_generate=False,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
        remove_unused_columns=False,
        save_safetensors=False,
        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
    
    stage1_trainer = CustomSeq2SeqTrainer(
        model=model,
        args=stage1_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=collator,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(patience=2)],
    )
    
    train_result = stage1_trainer.train()
    eval_results = stage1_trainer.evaluate()
    
    print(f"\n📊 نتائج المرحلة 1 النهائية:")
    for k, v in eval_results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
    
    model.check_audience_separation()
    
    print("✅ اكتملت المرحلة 1 - تم تدريب المصنّف")
    return model, stage1_trainer

def train_stage2_joint(model, trainer, train_ds, eval_ds, epochs=3):
    print("\n" + "="*60)
    print("🔄 المرحلة 2: التدريب المشترك (التلخيص + الجمهور)")
    print("="*60)
    
    for name, param in model.base.named_parameters():
        param.requires_grad = True
    
    trainer.args.num_train_epochs = epochs
    trainer.args.learning_rate = LR
    trainer.args.predict_with_generate = True
    trainer.args.compute_metrics = compute_metrics
    trainer.args.load_best_model_at_end = True
    trainer.args.metric_for_best_model = "eval_loss"
    trainer.args.greater_is_better = False
    
    trainer.optimizer = None
    trainer.create_optimizer()
    
    return trainer.train()

# ======================
# بدء التدريب
# ======================
print("\n🏁 بدء التدريب ثنائي المراحل (النسخة المُحسّنة)...")
print(f"📋 الإعدادات: LR={LR}, Classifier LR={LR_CLASSIFIER}, Base LR Multiplier={LR_BASE_MULTIPLIER}")
print(f"📋 قيم بيتا: SALIENCE={BETA_SALIENCE}, AUDIENCE={BETA_AUDIENCE}, DISTANCE={BETA_DISTANCE}")

# المرحلة 1
model, stage1_trainer = train_stage1_classifier(model, train_ds, eval_ds, epochs=3)

# حفظ نقطة التفتيش
stage1_output = f"{OUTPUT_DIR}/stage1-final"
model.save_pretrained(stage1_output)
tokenizer.save_pretrained(stage1_output)
print(f"💾 تم حفظ نقطة تفتيش المرحلة 1 في {stage1_output}")

# =====================================================
# عرض المرحلة 1
# =====================================================
print("\n" + "✨"*30)
print("🔍 عرض المرحلة 1: التنبؤ بنوع الجمهور")
print("✨"*30)

def demo_stage1_predict_audience(model, tokenizer, sample_texts=None):
    model.eval()
    model.to(device)
    
    if sample_texts is None:
        sample_texts = [
            "الشمس نجم يقع في مركز المجموعة الشمسية، وهي مصدر الضوء والحرارة للأرض.",
            "يحتوي الهاتف الذكي على معالج سريع، وشاشة عالية الدقة، وكاميرا متطورة.",
            "تُعد خوارزمية التعلم العميق نوعًا من الشبكات العصبية الاصطناعية."
        ]
    
    print("\n📋 النصوص للتنبؤ بالجمهور:")
    print("-" * 70)
    for i, text in enumerate(sample_texts, 1):
        print(f"\n[{i}] {text[:100]}...")
        pred_idx, probs = model.predict_audience(text, tokenizer, device)
        predicted_audience = AUDIENCE_TYPES[pred_idx]
        
        print(f"🎯 الجمهور المتوقع: {predicted_audience.upper()}")
        print(f"📊 توزيع الثقة:")
        for aud_idx, aud_name in enumerate(AUDIENCE_TYPES):
            bar = "█" * int(probs[aud_idx] * 30)
            print(f"   {aud_name:8s}: {probs[aud_idx]:5.2%} {bar}")
        print("-" * 70)
    
    return sample_texts

demo_stage1_predict_audience(model, tokenizer)

# المرحلة 2
trainer = CustomSeq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(patience=3)],
)

train_stage2_joint(model, trainer, train_ds, eval_ds, epochs=3)

# =====================================================
# عرض المرحلة 2
# =====================================================
print("\n" + "✨"*30)
print("🔄 عرض المرحلة 2: التنبؤ + التلخيص")
print("✨"*30)

def demo_stage2_predict_and_summarize(model, tokenizer, sample_texts=None):
    model.eval()
    model.to(device)
    
    if sample_texts is None:
        sample_texts = [
            "الذكاء الاصطناعي هو فرع من علوم الحاسوب يهدف إلى تطوير أنظمة قادرة على أداء مهام تتطلب ذكاءً بشريًا.",
            "تغير المناخ ظاهرة عالمية ناتجة عن زيادة انبعاثات غازات الدفيئة.",
            "التمثيل الضوئي هو العملية التي تستخدمها النباتات لتحويل الطاقة الضوئية إلى كيميائية."
        ]
    
    print("\n📋 النصوص للتنبؤ والتلخيص:")
    print("=" * 80)
    
    for i, text in enumerate(sample_texts, 1):
        print(f"\n🔹 النص [{i}]:\n{text[:150]}...\n")
        
        pred_idx, probs = model.predict_audience(text, tokenizer, device)
        detected_audience = AUDIENCE_TYPES[pred_idx]
        confidence = probs[pred_idx]
        
        print(f"🎯 الجمهور المكتشف: {detected_audience.upper()} (الثقة: {confidence:.2%})")
        
        prompt = AUDIENCE_TEMPLATES[detected_audience].format(text=text)
        inputs = tokenizer(
            prompt, 
            return_tensors="pt", 
            max_length=MAX_INPUT_LENGTH, 
            truncation=True,
            padding="max_length"
        ).to(device)
        
        gen_max_length = int(MAX_TARGET_LENGTH * AUDIENCE_LENGTH_TARGETS[detected_audience])
        
        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids, 
                attention_mask=inputs.attention_mask,
                max_length=gen_max_length,
                **GENERATION_CONFIG,
                decoder_start_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        summary = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        print(f"\n📝 الملخص ({detected_audience}):")
        print(f"   «{summary}»")
        print("=" * 80)
    
    return sample_texts

demo_stage2_predict_and_summarize(model, tokenizer)

# ======================
# التقييم
# ======================
def evaluate_audience_accuracy(model, tokenizer, audience_encoder, eval_dataset, device="cuda"):
    model.eval()
    if isinstance(device, str):
        device = torch.device(device)
    model.to(device)
    
    predictions, true_labels = [], []
    
    print(f"\n🔍 تقييم دقة التنبؤ بالجمهور على {len(eval_dataset)} عينة...")
    
    for sample in tqdm(eval_dataset, desc="جاري التقييم"):
        text = str(sample["text"]).strip()
        true_audience = str(sample["audience"]).strip()
        
        if not text or true_audience not in AUDIENCE_TYPES:
            continue
            
        pred_idx, probs = model.predict_audience(text, tokenizer, device)
        
        predictions.append(pred_idx)
        true_labels.append(audience_encoder.transform([true_audience])[0])

    predictions = np.array(predictions)
    true_labels = np.array(true_labels)
    
    if len(predictions) == 0:
        print("⚠️ لا توجد عينات صالحة للتقييم")
        return 0.0
    
    overall_accuracy = (predictions == true_labels).mean()
    
    print("\n📊 دقة التنبؤ لكل فئة:")
    print("-" * 50)
    for idx, aud_type in enumerate(AUDIENCE_TYPES):
        mask = true_labels == idx
        if mask.sum() > 0:
            acc = (predictions[mask] == idx).mean()
            print(f"  {aud_type:10s}: {acc:6.2%} ({mask.sum():3d} عينة)")
    
    print("-" * 50)
    print(f"✅ الدقة الإجمالية: {overall_accuracy:.2%}")
    
    print("\n📋 تقرير التصنيف:")
    print(classification_report(true_labels, predictions, target_names=AUDIENCE_TYPES, digits=4))
    
    cm = confusion_matrix(true_labels, predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=AUDIENCE_TYPES, yticklabels=AUDIENCE_TYPES)
    plt.title('مصفوفة الارتباك - التنبؤ بالجمهور')
    plt.ylabel('الفعلي')
    plt.xlabel('المتوقع')
    plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png')
    plt.close()
    print(f"\n💾 تم حفظ مصفوفة الارتباك في {OUTPUT_DIR}/confusion_matrix.png")
    
    return overall_accuracy

audience_acc = evaluate_audience_accuracy(
    model, 
    tokenizer, 
    audience_encoder, 
    eval_dataset=eval_ds_text,
    device=device
)

# ======================
# المقاييس النهائية
# ======================
print("\n" + "="*60)
print("📈 مقاييس التلخيص النهائية")
print("="*60)
metrics = trainer.evaluate()

if "eval_loss" in metrics and metrics["eval_loss"] is not None:
    metrics["perplexity"] = math.exp(metrics["eval_loss"])

for k, v in sorted(metrics.items()):
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")
print("="*60)

# ======================
# حفظ النموذج النهائي
# ======================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
joblib.dump(audience_encoder, os.path.join(OUTPUT_DIR, "audience_encoder.pkl"))

print(f"\n✅ تم حفظ النموذج والبيانات في: {OUTPUT_DIR}")

# فحص نهائي لتباعد الجمهور
print("\n" + "="*60)
print("📏 الفحص النهائي لتباعد تمثيلات الجمهور")
print("="*60)
final_separation = model.check_audience_separation()

# ======================
# دوال الاستدلال
# ======================
def summarize_for_audience(text, audience_type="adult", model=None, tokenizer=None, device="cuda"):
    if audience_type not in AUDIENCE_TYPES:
        raise ValueError(f"الجمهور يجب أن يكون أحد: {AUDIENCE_TYPES}")
    
    if isinstance(device, str):
        device = torch.device(device)
    model.eval()
    model.to(device)
    
    prompt = AUDIENCE_TEMPLATES[audience_type].format(text=text)
    
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        max_length=MAX_INPUT_LENGTH, 
        truncation=True,
        padding="max_length"
    ).to(device)
    
    gen_max_length = int(MAX_TARGET_LENGTH * AUDIENCE_LENGTH_TARGETS[audience_type])
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids, 
            attention_mask=inputs.attention_mask,
            max_length=gen_max_length,
            **GENERATION_CONFIG,
            decoder_start_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

def summarize_with_audience_detection(text, model=None, tokenizer=None, device="cuda"):
    if isinstance(device, str):
        device = torch.device(device)
    model.eval()
    model.to(device)
    
    pred_idx, probs = model.predict_audience(text, tokenizer, device)
    detected_audience = AUDIENCE_TYPES[pred_idx]
    confidence = probs[pred_idx]
    
    summary = summarize_for_audience(
        text, 
        audience_type=detected_audience, 
        model=model, 
        tokenizer=tokenizer, 
        device=device
    )
    
    return {
        "summary": summary,
        "detected_audience": detected_audience,
        "confidence": float(confidence),
        "audience_distribution": {
            aud: float(probs[i]) for i, aud in enumerate(AUDIENCE_TYPES)
        }
    }

# ======================
# الخاتمة
# ======================
print("\n" + "✨"*30)
print("🎉 اكتمل التدريب بنجاح!")
print("✨"*30)
print(f"\n📊 الدقة النهائية للجمهور: {audience_acc:.2%}")
print(f"📏 متوسط تباعد الجمهور: {final_separation:.3f}")
print(f"🎯 الهدف: دقة ≥90% | تباعد ≥0.3")
print(f"{'✅ تم تحقيق الهدف' if audience_acc >= 0.90 and final_separation >= 0.3 else '🔄 يحتاج لمزيد من الضبط'}")

print("\n📚 أمثلة الاستخدام:")
print('   # التنبؤ بالجمهور فقط:')
print('   pred_idx, probs = model.predict_audience(text, tokenizer)')
print()
print('   # التلخيص مع التنبؤ:')
print('   result = summarize_with_audience_detection(text, model, tokenizer)')
print('   print(result["summary"])')
print()
print('   # التلخيص لجمهور محدد:')
print('   summary = summarize_for_audience(text, "kids", model, tokenizer)')